# Runtime y Framework

Provider configura dónde se ejecuta un modelo; framework configura el loop del Agent. Backend configura su flujo.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [ ]:
import agentic_systems as toolkit


In [ ]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

In [ ]:
for name in ("native", "langgraph", "openai-agents", "strands"):
    agent = toolkit.Agent(
        name="operator", runtime=toolkit.runtime(provider="python-runtime"),
        framework=toolkit.framework(name), skills=(arithmetic,),
    )
    agent.compile()  # prepara sin instanciar ni llamar al proveedor
    result = agent.run({"tool": "increment", "input": {"value": 3}})
    assert result.ok and result.data["value"] == 4
    print(name, result.data)

El mismo constructor Agent sirve para un flujo o para un actor con runtime explícito. Para incluir razonamiento en un flujo, coloca ese Agent como uno de sus pasos. No se infiere un proveedor ni se permite fallback automático. Las siguientes configuraciones **se declaran y no se ejecutan**: no acreditan compatibilidad live.

Los límites internos del actor se declaran en runtime. Limits limita el flujo completo. framework recibe opciones nativas mediante agent_kwargs y run_kwargs; las claves del ag no pueden sobrescribirse. Tener un SDK instalado o configurado no acredita un proveedor live.

In [ ]:
configured = toolkit.runtime(provider='bedrock-runtime',model='explicit-configured-only',region='us-east-1',endpoint='https://fixture.invalid',api_key='fixture-secret',timeout_s=30,max_retries=1,max_tool_calls=3,max_turns=8,max_tokens=128,backoff_s=0.1,max_concurrency=2)
assert 'fixture-secret' not in str(configured.inspect())
print(configured.inspect())
options = toolkit.framework('openai-agents',agent_kwargs={'handoff_description':'configured-only'},run_kwargs={'context':{'tenant':'fixture'}})
print(options.inspect())

In [ ]:
configured_only = {
    name: toolkit.runtime(provider=name)
    for name in ("openai-runtime", "ollama-runtime", "bedrock-runtime", "vllm-runtime")
}
assert set(configured_only) == {"openai-runtime", "ollama-runtime", "bedrock-runtime", "vllm-runtime"}
print({name: "configurado; no ejecutado" for name in configured_only})